# SafeDrive: VS Code + Colab connection test

Run this notebook from top to bottom using a Colab kernel in VS Code. It verifies five things without starting RL training:

1. VS Code can execute on a Colab-managed runtime.
2. PyTorch can actually use the assigned GPU.
3. Google Drive can be mounted and written to.
4. The same SafeDrive revision can be cloned from GitHub and installed.
5. MetaDrive can run headlessly on the remote machine.

Authentication prompts must be completed by you. Do not paste Google passwords or GitHub tokens into this notebook.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/djdhillxn/safedrive.git"
REPO_DIR = Path("/content/safedrive")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT = DRIVE_MOUNT / "MyDrive" / "SafeDrive"

# Set this to True only if GitHub requires authentication to clone the repository.
PRIVATE_REPO = False

print(f"Remote workspace: {REPO_DIR}")
print(f"Persistent storage: {DRIVE_PROJECT}")

In [ ]:
import platform
import subprocess
import sys
import time

import torch

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
subprocess.run(["nvidia-smi"], check=False)

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is attached. In VS Code choose Select Kernel > Colab > "
        "New Colab Server and select a GPU machine."
    )

gpu_name = torch.cuda.get_device_name(0)
start = time.perf_counter()
left = torch.rand((1024, 1024), device="cuda")
right = torch.rand((1024, 1024), device="cuda")
product = left @ right
torch.cuda.synchronize()
elapsed = time.perf_counter() - start
assert product.is_cuda
print(f"GPU calculation passed on {gpu_name} in {elapsed:.3f} seconds.")

In [ ]:
from datetime import datetime, timezone

from google.colab import drive

# Google will display a separate authorization prompt for this mount.
drive.mount(str(DRIVE_MOUNT))
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)

drive_probe = DRIVE_PROJECT / "colab_connection_test.txt"
probe_text = f"SafeDrive Colab connection passed at {datetime.now(timezone.utc).isoformat()}\n"
drive_probe.write_text(probe_text, encoding="utf-8")
assert drive_probe.read_text(encoding="utf-8") == probe_text
print(f"Google Drive read/write passed: {drive_probe}")

## GitHub access

For a public repository, leave `PRIVATE_REPO = False`. For a private repository, first add a read-only fine-grained token named `GITHUB_TOKEN` in Colab's Secrets panel, grant this notebook access to the secret, and set `PRIVATE_REPO = True`. The token is passed to Git only for authentication and is never written into the repository's remote URL.

In [ ]:
import os

git_environment = os.environ.copy()
askpass_path = None

if PRIVATE_REPO:
    from google.colab import userdata

    github_token = userdata.get("GITHUB_TOKEN")
    if not github_token:
        raise RuntimeError("Colab secret GITHUB_TOKEN is missing or not enabled.")

    askpass_path = Path("/tmp/safedrive_git_askpass.sh")
    askpass_path.write_text(
        '#!/bin/sh\ncase "$1" in\n'
        '*Username*) printf "%s\\n" "x-access-token" ;;\n'
        '*Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        "esac\n",
        encoding="utf-8",
    )
    askpass_path.chmod(0o700)
    git_environment["GIT_ASKPASS"] = str(askpass_path)
    git_environment["GIT_TERMINAL_PROMPT"] = "0"
    git_environment["GITHUB_TOKEN"] = github_token

try:
    if (REPO_DIR / ".git").exists():
        git_command = ["git", "-C", str(REPO_DIR), "pull", "--ff-only"]
    elif REPO_DIR.exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
    else:
        git_command = ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)]
    subprocess.run(git_command, check=True, env=git_environment)
finally:
    if askpass_path is not None:
        askpass_path.unlink(missing_ok=True)

os.chdir(REPO_DIR)
git_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()
print(f"SafeDrive revision: {git_commit}")
print(f"Working directory: {Path.cwd()}")

In [ ]:
import importlib.metadata

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--progress-bar",
        "off",
        "-e",
        str(REPO_DIR),
    ],
    check=True,
)

package_names = [
    "metadrive-simulator",
    "stable-baselines3",
    "sb3-contrib",
    "gymnasium",
]
package_versions = {
    name: importlib.metadata.version(name) for name in package_names
}
for name, version in package_versions.items():
    print(f"{name}: {version}")

import saferl_drive
from stable_baselines3 import PPO, SAC

print(f"SafeDrive import passed: {saferl_drive.__version__}")

In [ ]:
from metadrive.envs import MetaDriveEnv

environment = MetaDriveEnv(
    {
        "use_render": False,
        "log_level": 50,
        "num_scenarios": 1,
        "start_seed": 0,
        "traffic_density": 0.0,
        "random_traffic": False,
        "horizon": 50,
    }
)

rollout_reward = 0.0
rollout_steps = 0
try:
    observation, info = environment.reset(seed=0)
    for _ in range(25):
        action = environment.action_space.sample()
        observation, reward, terminated, truncated, info = environment.step(action)
        rollout_reward += float(reward)
        rollout_steps += 1
        if terminated or truncated:
            break
finally:
    environment.close()

assert rollout_steps > 0
print(
    f"Headless MetaDrive rollout passed: {rollout_steps} steps, "
    f"reward {rollout_reward:.3f}."
)

In [ ]:
import json

report = {
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "git_commit": git_commit,
    "gpu": gpu_name,
    "python": sys.version.split()[0],
    "packages": package_versions,
    "metadrive_steps": rollout_steps,
    "metadrive_reward": rollout_reward,
}
report_dir = DRIVE_PROJECT / "smoke_tests"
report_dir.mkdir(parents=True, exist_ok=True)
report_name = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S.json")
report_path = report_dir / report_name
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
print(f"\nAll checks passed. Persistent report: {report_path}")

## Finished

If the report cell succeeded, VS Code, the Colab GPU runtime, GitHub, Google Drive, SafeDrive, and headless MetaDrive are connected correctly. Disconnect the Colab server when finished so it does not continue consuming compute units.

For later sessions, push local code changes to GitHub and rerun this notebook's clone/pull cell. Keep active training files on `/content` for speed, then copy checkpoints and completed runs to `MyDrive/SafeDrive`; do not treat the mounted Drive folder as a live Git checkout.